In [2]:
"""
exercise4.py - Exercise 4: System of ODEs
System: dy0/dx = -y0 - 3x, dy1/dx = -y1 - 3x
y0(0) = 5, y1(0) = 6, x ∈ [0, 2]
"""

import numpy as np

# ============================================
# RK3 FUNCTION
# ============================================
def rk3(f_ode, xRange, yInitial, numSteps):
    """
    x, y = rk3(f_ode, xRange, yInitial, numSteps)
    
    3rd order Runge-Kutta method (RK3) for solving ODEs.
    Works for both scalar and system of ODEs.
    """
    x = np.zeros(numSteps + 1)
    y = np.zeros((numSteps + 1, np.size(yInitial)))
    dx = (xRange[1] - xRange[0]) / numSteps
    
    for k in range(0, numSteps + 1):
        if k == 0:
            x[0] = xRange[0]
            y[0, :] = yInitial
        else:
            # First intermediate point (midpoint)
            xa = x[k-1] + dx/2
            ya = y[k-1, :] + (dx/2) * f_ode(x[k-1], y[k-1, :])
            
            # Second intermediate point (endpoint with correction)
            xb = x[k-1] + dx
            yb = y[k-1, :] + dx * (2*f_ode(xa, ya) - f_ode(x[k-1], y[k-1, :]))
            
            # Final step using weighted average
            x[k] = x[k-1] + dx
            y[k, :] = y[k-1, :] + (dx/6) * (
                f_ode(x[k-1], y[k-1, :]) + 
                4*f_ode(xa, ya) + 
                f_ode(xb, yb)
            )
    
    if y.shape[1] == 1:
        y = y.flatten()
    
    return x, y


# ============================================
# ODE FUNCTIONS
# ============================================

def expm_ode_system(x, y):
    """
    System of ODEs:
    dy0/dx = -y0 - 3x
    dy1/dx = -y1 - 3x
    
    Input:
    x : independent variable
    y : array [y0, y1]
    
    Returns:
    array [dy0/dx, dy1/dx]
    """
    dy0 = -y[0] - 3.0 * x
    dy1 = -y[1] - 3.0 * x
    return np.array([dy0, dy1])


def expm_ode_scalar(x, y):
    """
    Scalar ODE: dy/dx = -y - 3x
    """
    return -y - 3.0 * x


def exact_solution(x, y0):
    """
    Exact solution for dy/dx = -y - 3x, y(0) = y0
    y(x) = (y0 + 3)*e^(-x) - 3x - 2e^(-x) + 3? Wait, let me derive.
    
    Actually: dy/dx = -y - 3x
    Solution: y(x) = (y0 + 3)e^(-x) - 3x + 3? No.
    
    Let me give the correct exact solution:
    For dy/dx = -y - 3x, y(0) = y0
    y(x) = (y0 + 3)*e^(-x) - 3x + 3? Let's check at x=0: y(0)=y0+3-0+3? That gives y0+6. Wrong.
    
    Correct: y(x) = (y0 + 3)*e^(-x) - 3x + 3? Still wrong.
    
    Actually the exact solution is: y(x) = (y0 + 3)e^(-x) - 3x + 3? Let me verify.
    
    Better to compute numerically and compare with system solution.
    """
    # For dy/dx = -y - 3x, y(0) = y0
    # The exact solution is: y(x) = y0*e^(-x) + 3e^(-x) - 3x + 3? No.
    
    # Let me derive properly:
    # dy/dx + y = -3x
    # Integrating factor: e^x
    # d/dx (y e^x) = -3x e^x
    # y e^x = -3∫ x e^x dx = -3(x e^x - e^x) + C
    # y e^x = -3x e^x + 3e^x + C
    # y = -3x + 3 + C e^(-x)
    # At x=0: y(0) = 3 + C = y0 → C = y0 - 3
    # y(x) = (y0 - 3)e^(-x) - 3x + 3
    
    return (y0 - 3) * np.exp(-x) - 3 * x + 3


# ============================================
# MAIN CODE
# ============================================

print("="*80)
print("EXERCISE 4: SYSTEM OF ODEs")
print("System: dy0/dx = -y0 - 3x, dy1/dx = -y1 - 3x")
print("Initial conditions: y0(0) = 5, y1(0) = 6")
print("x ∈ [0, 2], numSteps = 40")
print("="*80)

# Parameters
xRange = np.array([0.0, 2.0])
yInitial_system = np.array([5.0, 6.0])  # [y0, y1]
numSteps = 40

# ============================================
# PART 1: Solve the system using RK3
# ============================================
print("\n" + "-"*50)
print("PART 1: Solving the system with RK3")
print("-"*50)

xs, ys = rk3(expm_ode_system, xRange, yInitial_system, numSteps)

# Final values
final_x = xs[-1]
final_y = ys[-1, :]

print(f"\nFinal x: {final_x}")
print(f"Final y values: [{final_y[0]:.10f}, {final_y[1]:.10f}]")

# ============================================
# PART 2: Solve scalar ODE twice
# ============================================
print("\n" + "-"*50)
print("PART 2: Solving scalar ODE twice (y0=5 and y0=6)")
print("-"*50)

# Solve with y(0) = 5
x0, y0 = rk3(expm_ode_scalar, xRange, 5.0, numSteps)

# Solve with y(0) = 6
x1, y1 = rk3(expm_ode_scalar, xRange, 6.0, numSteps)

print(f"\nScalar solution with y(0)=5: final y = {y0[-1]:.10f}")
print(f"Scalar solution with y(0)=6: final y = {y1[-1]:.10f}")

# ============================================
# PART 3: Compare results
# ============================================
print("\n" + "-"*50)
print("PART 3: COMPARISON")
print("-"*50)

print("\nComparing last entries:")
print(f"  System ys[-1,0] = {final_y[0]:.10f}")
print(f"  Scalar y0[-1]    = {y0[-1]:.10f}")
print(f"  Difference:      = {abs(final_y[0] - y0[-1]):.10e}")
print()
print(f"  System ys[-1,1] = {final_y[1]:.10f}")
print(f"  Scalar y1[-1]    = {y1[-1]:.10f}")
print(f"  Difference:      = {abs(final_y[1] - y1[-1]):.10e}")

# Check if they match
tol = 1e-10
if abs(final_y[0] - y0[-1]) < tol and abs(final_y[1] - y1[-1]) < tol:
    print("\n✓ Results match! System solution = two independent scalar solutions.")
else:
    print("\n✗ Results do not match. Check your code.")

# ============================================
# PART 4: Norm of differences for all entries
# ============================================
print("\n" + "-"*50)
print("PART 4: NORM OF DIFFERENCES (all entries)")
print("-"*50)

# Compare y0 from system vs scalar
diff0 = ys[:, 0] - y0
diff1 = ys[:, 1] - y1

norm0 = np.linalg.norm(diff0)
norm1 = np.linalg.norm(diff1)

print(f"\nNorm of difference for y0 (system vs scalar): {norm0:.10e}")
print(f"Norm of difference for y1 (system vs scalar): {norm1:.10e}")

if norm0 < tol and norm1 < tol:
    print("\n✓ All entries match! The two methods produce identical results.")
else:
    print("\n✗ Some differences exist.")

# ============================================
# EXTRA: Compare with exact solution
# ============================================
print("\n" + "-"*50)
print("EXTRA: Comparison with exact solution")
print("-"*50)

exact_5 = exact_solution(2.0, 5.0)
exact_6 = exact_solution(2.0, 6.0)

print(f"\nExact solution at x=2, y(0)=5: {exact_5:.10f}")
print(f"RK3 solution (scalar) at x=2: {y0[-1]:.10f}")
print(f"Error: {abs(y0[-1] - exact_5):.10e}")
print()
print(f"Exact solution at x=2, y(0)=6: {exact_6:.10f}")
print(f"RK3 solution (scalar) at x=2: {y1[-1]:.10f}")
print(f"Error: {abs(y1[-1] - exact_6):.10e}")

# ============================================
# PRINT FINAL ANSWER (as requested)
# ============================================
print("\n" + "="*80)
print("FINAL ANSWER :")
print("="*80)
print(f"\nxs[-1] = {xs[-1]}")
print(f"ys[-1,:] = [{ys[-1,0]:.10f}, {ys[-1,1]:.10f}]")
print("\nComparison:")
print(f"  ys[-1,0] = {ys[-1,0]:.10f} vs y0[-1] = {y0[-1]:.10f}")
print(f"  ys[-1,1] = {ys[-1,1]:.10f} vs y1[-1] = {y1[-1]:.10f}")
print("\n" + "="*80)

EXERCISE 4: SYSTEM OF ODEs
System: dy0/dx = -y0 - 3x, dy1/dx = -y1 - 3x
Initial conditions: y0(0) = 5, y1(0) = 6
x ∈ [0, 2], numSteps = 40

--------------------------------------------------
PART 1: Solving the system with RK3
--------------------------------------------------

Final x: 2.000000000000001
Final y values: [-2.7293323682, -2.5939985522]

--------------------------------------------------
PART 2: Solving scalar ODE twice (y0=5 and y0=6)
--------------------------------------------------

Scalar solution with y(0)=5: final y = -2.7293323682
Scalar solution with y(0)=6: final y = -2.5939985522

--------------------------------------------------
PART 3: COMPARISON
--------------------------------------------------

Comparing last entries:
  System ys[-1,0] = -2.7293323682
  Scalar y0[-1]    = -2.7293323682
  Difference:      = 0.0000000000e+00

  System ys[-1,1] = -2.5939985522
  Scalar y1[-1]    = -2.5939985522
  Difference:      = 0.0000000000e+00

✓ Results match! System s